In [21]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [22]:
NEW_DATA_ROOT = "/kaggle/input/datasets/ichbingiangthanh/crop-data-dongvan-copy/crop_data_dongvan - Copy"#dongvan dataset
OLD_DATA_ROOT = "/kaggle/input/datasets/ichbingiangthanh/vnua-dataset-fullnamespieces-080626/crop_data - filter_250126 - Full name/crop_data" #vnua dataset                  

TAX_VNUA_PATH  = "/kaggle/input/datasets/ichbingiangthanh/id-vnua-080626/id_vnua.xlsx"
TAX_VNUA_SHEET = "Sheet3"

TAX_DV_PATH    = "/kaggle/input/datasets/ichbingiangthanh/id-dongvan-fullname-090626/dongvan_data_2.xlsx"
TAX_DV_SHEET   = "data_dongvan_id"

SEED = 42
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
REPLAY_RATIO = 0.5         
REPLAY_FROM_ALL_OLD = True # True: replay từ mọi loài VNUA; False: chỉ replay các loài có trong DV

BATCH_SIZE_TRAIN = 32
BATCH_SIZE_EVAL  = 64

IMG_EXT = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp"}
rng = np.random.RandomState(SEED)


def norm(s):
    s = str(s).strip()
    s = " ".join(s.split())  
    return s

In [ ]:
# read taxonomy excel file

def read_tax_vnua(path, sheet=None):
    df = pd.read_excel(path, sheet_name=sheet) if sheet is not None else pd.read_excel(path)
    df.columns = [c.strip().lower() for c in df.columns]
    rename_map = {}
    if "loài" in df.columns: rename_map["loài"] = "species"
    if "chi"  in df.columns: rename_map["chi"]  = "genus"
    if "họ"   in df.columns: rename_map["họ"]   = "family"
    if "bộ"   in df.columns: rename_map["bộ"]   = "order"
    df = df.rename(columns=rename_map)

    need = ["species","genus","family","order"]
    miss = [c for c in need if c not in df.columns]
    assert not miss, f"Taxonomy VNUA thiếu cột {miss}."

    df = df[need].copy()
    for c in ["genus","family","order"]:
        df[c] = df[c].map(norm)
    df["species"] = df["species"].astype(str).str.strip().str.lower()
    return df.dropna().drop_duplicates()

def read_tax_dv(path, sheet=None):
    df = pd.read_excel(path, sheet_name=sheet) if sheet is not None else pd.read_excel(path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "tên khoa học" in df.columns: df = df.rename(columns={"tên khoa học":"species"})

    need = ["species","order","family","genus"]
    miss = [c for c in need if c not in df.columns]
    assert not miss, f"Taxonomy Dong Van thiếu cột {miss}"

    df = df[need].copy()
    for c in ["genus","family","order"]:
        df[c] = df[c].map(norm)
    df["species"] = df["species"].astype(str).str.strip().str.lower()  
    return df

tax_vnua = read_tax_vnua(TAX_VNUA_PATH, sheet=TAX_VNUA_SHEET)
tax_dv   = read_tax_dv(TAX_DV_PATH, sheet=TAX_DV_SHEET)
SPCODE_MAP = {}

for sp in tax_dv["species"].dropna().unique():
    sp = str(sp).strip().lower()
    SPCODE_MAP[sp[:4]] = sp

def map_spcode(code):
    return SPCODE_MAP.get(code, code)

tax_all = pd.concat([tax_vnua, tax_dv], axis=0, ignore_index=True).drop_duplicates()
print("Tax rows | vnua spiece:", len(tax_vnua), "dv spiece:", len(tax_dv), "merged:", len(tax_all))

species_to_genus = dict(zip(tax_all["species"], tax_all["genus"]))
genus_to_family  = dict(zip(tax_all["genus"],  tax_all["family"]))
family_to_order  = dict(zip(tax_all["family"], tax_all["order"]))

# LOAD label maps from checkpoint VNUA base model

pt_candidates = glob.glob("/kaggle/input/models/ichbingiangthanh/model-vnua-080626/pytorch/default/1/trained_model_vnua_base_080626.pt", recursive=True)
if len(pt_candidates) == 0:
    pt_candidates = glob.glob("/kaggle/input/**/*.pt", recursive=True)
assert len(pt_candidates) > 0, "Không tìm thấy model"
CKPT_PATH = pt_candidates[0]
ckpt = torch.load(CKPT_PATH, map_location="cpu")

order2idx   = ckpt["order2idx"]
family2idx  = ckpt["family2idx"]
genus2idx   = ckpt["genus2idx"]
species2idx = ckpt["species2idx"]

order2idx_n  = {norm(k): v for k,v in order2idx.items()}
family2idx_n = {norm(k): v for k,v in family2idx.items()}
genus2idx_n  = {norm(k): v for k,v in genus2idx.items()}
species2idx_n= {str(k).strip().lower(): v for k,v in species2idx.items()}

# 3) labeling spiece name for each image in dongvan dataset
def folder_to_spcode(folder_name: str) -> str:
    s = folder_name.strip().lower()
    return s[:4]

def collect_items_with_spcode(root_dir):
    root = Path(root_dir)
    items = []
    missing_folders = []
    for sp_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        sp_full = map_spcode(folder_to_spcode(sp_dir.name))
        if sp_full not in species_to_genus:
            missing_folders.append((sp_dir.name, sp_code))
            continue
        for img in sp_dir.rglob("*"):
            if img.is_file() and img.suffix.lower() in IMG_EXT:
                items.append((str(img), sp_full))
    return items, missing_folders

new_items, missing_tax_folders = collect_items_with_spcode(NEW_DATA_ROOT)
assert len(new_items) > 0, "Không tìm thấy ảnh"

print("DongVan dataset total images:", len(new_items))
if len(missing_tax_folders) > 0:
    print("folders missing taxonomy (skipped):", len(missing_tax_folders))
    print("missing:", missing_tax_folders[:10])

# 4) tran/val/test split on dongvan dataset

idx = np.arange(len(new_items))
rng.shuffle(idx)

n = len(new_items)

train_n = int(TRAIN_RATIO * n)
val_n   = int(VAL_RATIO * n)

# phần còn lại cho test
test_n = n - train_n - val_n

train_idx = idx[:train_n]
val_idx   = idx[train_n:train_n + val_n]
test_idx  = idx[train_n + val_n:]

train_new_items = [new_items[i] for i in train_idx]
val_new_items   = [new_items[i] for i in val_idx]
test_new_items  = [new_items[i] for i in test_idx]

print("Train:", len(train_new_items))
print("Val:", len(val_new_items))
print("Test:", len(test_new_items))

# 5) Replay from VNUA dataset (only for training)

def collect_old_items(root_dir, dv_codes=None, from_all=True):
    root = Path(root_dir)
    items = []

    for sp_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        sp_full = sp_dir.name.strip().lower()

        if sp_full not in species_to_genus:
            continue

        if sp_full not in species2idx_n:
            continue

        if (not from_all) and (dv_codes is not None):
            if sp_full not in dv_codes:
                continue

        for img in sp_dir.rglob("*"):
            if img.is_file() and img.suffix.lower() in IMG_EXT:
                items.append((str(img), sp_full))

    return items


# lấy mã loài từ DongVan
dv_codes = set([sp for _, sp in new_items])

old_pool = collect_old_items(
    OLD_DATA_ROOT,
    dv_codes=dv_codes,
    from_all=REPLAY_FROM_ALL_OLD
)

rng.shuffle(old_pool)

# replay chỉ cộng vào train
replay_n = min(
    len(old_pool),
    int(len(train_new_items) * REPLAY_RATIO)
)

replay_items = old_pool[:replay_n]

train_items = train_new_items + replay_items
rng.shuffle(train_items)

# val/test giữ nguyên
val_items = val_new_items
test_items = test_new_items


print(
    f"Train={len(train_items)} "
    f"(DV={len(train_new_items)} + replay={len(replay_items)})"
)
print("Val =", len(val_items))
print("Test =", len(test_items))

# 6) # Tạo dataset phân cấp với cơ chế fallback nhãn (species --> genus --> family --> order)
#label và mask để huấn luyện khi thiếu nhãn chi tiết.
#example:
# tensor ảnh:
#3  order
#12,  family
#50,  genus
#-1,  pecies thiếu
#--> mask[1,1,1,0]

def compute_masks_and_labels(sp_key: str):

    g = species_to_genus[sp_key]             
    f = genus_to_family[g]
    o = family_to_order[f]

    y_o = order2idx_n.get(norm(o), -1)
    y_f = family2idx_n.get(norm(f), -1)
    y_g = genus2idx_n.get(norm(g), -1)
    y_s = species2idx_n.get(sp_key, -1)

    o_ok = (y_o != -1)
    f_ok = (y_f != -1)
    g_ok = (y_g != -1)
    s_ok = (y_s != -1)

    if s_ok:
        return y_o, y_f, y_g, y_s, int(o_ok), int(f_ok), int(g_ok), 1
    if g_ok:
        return y_o, y_f, y_g, -1, int(o_ok), int(f_ok), 1, 0
    if f_ok:
        return y_o, y_f, -1, -1, int(o_ok), 1, 0, 0
    if o_ok:
        return y_o, -1, -1, -1, 1, 0, 0, 0
    return -1, -1, -1, -1, 0, 0, 0, 0

class FallbackHier4Dataset(Dataset):
    def __init__(self, items, tf):
        self.tf = tf
        self.cache = []
        self.stats = {"species":0,"genus":0,"family":0,"order":0,"skip":0}

        for path, sp in items:
            sp = str(sp).strip().lower()

            # DV code có thể cần alias
            sp = str(sp).strip().lower()

            if sp not in species_to_genus:
                self.stats["skip"] += 1
                continue

            y_o,y_f,y_g,y_s,m_o,m_f,m_g,m_s = compute_masks_and_labels(sp)
            if (m_o+m_f+m_g+m_s) == 0:
                self.stats["skip"] += 1
                continue

            if m_s == 1: self.stats["species"] += 1
            elif m_g == 1: self.stats["genus"] += 1
            elif m_f == 1: self.stats["family"] += 1
            elif m_o == 1: self.stats["order"] += 1

            self.cache.append((path, sp, y_o,y_f,y_g,y_s, m_o,m_f,m_g,m_s))

        assert len(self.cache) > 0, "Không còn mẫu hợp lệ sau fallback (check taxonomy/overlap)."
        print("[Dataset stats]", self.stats)

    def __len__(self): return len(self.cache)

    def __getitem__(self, i):
        path, sp, y_o,y_f,y_g,y_s, m_o,m_f,m_g,m_s = self.cache[i]
        img = Image.open(path).convert("RGB")
        x = self.tf(img)

        
        y_o = 0 if y_o < 0 else y_o
        y_f = 0 if y_f < 0 else y_f
        y_g = 0 if y_g < 0 else y_g
        y_s = 0 if y_s < 0 else y_s

        mask = torch.tensor([m_o,m_f,m_g,m_s], dtype=torch.float32)
        return x, torch.tensor(y_o), torch.tensor(y_f), torch.tensor(y_g), torch.tensor(y_s), mask

# 7) Transforms + Loaders

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.35,
        contrast=0.35,
        saturation=0.25,
        hue=0.05
    ),
    transforms.RandomGrayscale(p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    ),
])

eval_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    ),
])


# Dataset
train_ds = FallbackHier4Dataset(train_items, train_tf)
val_ds   = FallbackHier4Dataset(val_items, eval_tf)
test_ds  = FallbackHier4Dataset(test_items, eval_tf)

# Loaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE_EVAL,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Prepared loaders:",
    "train=", len(train_ds),
    "val=", len(val_ds),
    "test=", len(test_ds),
    "ckpt=", CKPT_PATH
)

#Student training with Branch-Masked Hierarchical KD
import copy
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

assert "species_to_genus" in globals(), "Thiếu species_to_genus."
assert "genus_to_family" in globals(), "Thiếu genus_to_family."
assert "family_to_order" in globals(), "Thiếu family_to_order."
assert "order2idx_n" in globals(), "Thiếu order2idx_n."
assert "family2idx_n" in globals(), "Thiếu family2idx_n."
assert "genus2idx_n" in globals(), "Thiếu genus2idx_n."
assert "species2idx_n" in globals(), "Thiếu species2idx_n."

# 1) Build taxonomy children index maps for branch masking

order_to_family_idx = defaultdict(list)
family_to_genus_idx = defaultdict(list)
genus_to_species_idx = defaultdict(list)

for sp, g in species_to_genus.items():
    sp = str(sp).strip().lower()
    g = str(g).strip()
    if g not in genus_to_family:
        continue
    f = str(genus_to_family[g]).strip()
    if f not in family_to_order:
        continue
    o = str(family_to_order[f]).strip()

    # order -> family
    if o in order2idx_n and f in family2idx_n:
        order_to_family_idx[order2idx_n[o]].append(family2idx_n[f])

    # family -> genus
    if f in family2idx_n and g in genus2idx_n:
        family_to_genus_idx[family2idx_n[f]].append(genus2idx_n[g])

    # genus -> species
    if g in genus2idx_n and sp in species2idx_n:
        genus_to_species_idx[genus2idx_n[g]].append(species2idx_n[sp])

# unique + sort
for d in [order_to_family_idx, family_to_genus_idx, genus_to_species_idx]:
    for k in list(d.keys()):
        d[k] = sorted(list(set(d[k])))

print("Branch map stats:")
print("  orders with family children :", len(order_to_family_idx))
print("  families with genus children:", len(family_to_genus_idx))
print("  genera with species children:", len(genus_to_species_idx))

# Model definition
class Hier4Net(nn.Module):
    def __init__(self, n_order, n_family, n_genus, n_species):
        super().__init__()
        backbone = models.resnet18(weights=None)
        d = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.drop = nn.Dropout(0.2)
        self.head_o = nn.Linear(d, n_order)
        self.head_f = nn.Linear(d, n_family)
        self.head_g = nn.Linear(d, n_genus)
        self.head_s = nn.Linear(d, n_species)

    def forward(self, x):
        feat = self.drop(self.backbone(x))
        return self.head_o(feat), self.head_f(feat), self.head_g(feat), self.head_s(feat)
# 3) Load checkpoint and create teacher/student
ckpt = torch.load(CKPT_PATH, map_location="cpu")

student = Hier4Net(
    n_order=len(order2idx_n),
    n_family=len(family2idx_n),
    n_genus=len(genus2idx_n),
    n_species=len(species2idx_n),
).to(DEVICE)

student.load_state_dict(ckpt["model"], strict=True)

teacher = copy.deepcopy(student).to(DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher/student from:", CKPT_PATH)

# 4) Freeze strategy: freeze all backbone, unfreeze layer4 + heads
for p in student.backbone.parameters():
    p.requires_grad = False

for name, p in student.backbone.named_parameters():
    if "layer4" in name:
        p.requires_grad = True

for head in [student.head_o, student.head_f, student.head_g, student.head_s]:
    for p in head.parameters():
        p.requires_grad = True

trainable_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in student.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,}")

# 5) Loss helpers
def kd_kl(student_logits, teacher_logits, T=2.0):
    ps = F.log_softmax(student_logits / T, dim=1)
    pt = F.softmax(teacher_logits / T, dim=1)
    return F.kl_div(ps, pt, reduction="batchmean") * (T * T)

def branch_ce_loss(batch_logits, batch_targets, parent_targets, child_map):
    losses = []
    B = batch_logits.size(0)

    for i in range(B):
        tgt = int(batch_targets[i].item())
        par = int(parent_targets[i].item())

        allowed = child_map.get(par, None)
        if allowed is None or len(allowed) == 0:
            continue
        if tgt not in allowed:
            continue

        allowed_t = torch.tensor(allowed, dtype=torch.long, device=batch_logits.device)
        sub_logits = batch_logits[i:i+1, allowed_t]  # [1, K]
        local_tgt = torch.tensor([allowed.index(tgt)], dtype=torch.long, device=batch_logits.device)
        losses.append(F.cross_entropy(sub_logits, local_tgt))

    if len(losses) == 0:
        return torch.tensor(0.0, device=batch_logits.device)
    return torch.stack(losses).mean()

def branch_kd_loss(batch_s_logits, batch_t_logits, parent_targets, child_map, T=2.0):
    losses = []
    B = batch_s_logits.size(0)

    for i in range(B):
        par = int(parent_targets[i].item())
        allowed = child_map.get(par, None)
        if allowed is None or len(allowed) == 0:
            continue

        allowed_t = torch.tensor(allowed, dtype=torch.long, device=batch_s_logits.device)
        s_sub = batch_s_logits[i:i+1, allowed_t]
        t_sub = batch_t_logits[i:i+1, allowed_t]
        losses.append(kd_kl(s_sub, t_sub, T=T))

    if len(losses) == 0:
        return torch.tensor(0.0, device=batch_s_logits.device)
    return torch.stack(losses).mean()

def safe_ce(logits, targets, valid_mask):
    if valid_mask.sum() <= 0:
        return torch.tensor(0.0, device=logits.device)
    return F.cross_entropy(logits[valid_mask], targets[valid_mask])

def safe_kd(s_logits, t_logits, valid_mask, T=2.0):
    if valid_mask.sum() <= 0:
        return torch.tensor(0.0, device=s_logits.device)
    return kd_kl(s_logits[valid_mask], t_logits[valid_mask], T=T)
# 6) Hyperparameters

W_O, W_F, W_G, W_S = 0.4, 0.6, 0.8, 1.0
LAMBDA_KD = 1.0
TEMP = 2.0

LR = 1e-4
WD = 1e-4
EPOCHS = 8

SAVE_PATH = "/kaggle/working/model_distillation_080626.pt"

opt = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, student.parameters()),
    lr=LR,
    weight_decay=WD
)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

# 7) Evaluation
@torch.no_grad()
def eval_best_available(model, loader):
    model.eval()
    tot = 0
    correct = 0

    for x, y_o, y_f, y_g, y_s, mask in loader:
        x = x.to(DEVICE)
        y_o = y_o.to(DEVICE)
        y_f = y_f.to(DEVICE)
        y_g = y_g.to(DEVICE)
        y_s = y_s.to(DEVICE)
        mask = mask.to(DEVICE)

        lo, lf, lg, ls = model(x)
        po = lo.argmax(1)
        pf = lf.argmax(1)
        pg = lg.argmax(1)
        ps = ls.argmax(1)

        use_s = mask[:, 3] > 0.5
        use_g = (mask[:, 2] > 0.5) & (~use_s)
        use_f = (mask[:, 1] > 0.5) & (~use_s) & (~use_g)
        use_o = (mask[:, 0] > 0.5) & (~use_s) & (~use_g) & (~use_f)

        corr = torch.zeros(x.size(0), dtype=torch.bool, device=DEVICE)
        corr[use_s] = (ps[use_s] == y_s[use_s])
        corr[use_g] = (pg[use_g] == y_g[use_g])
        corr[use_f] = (pf[use_f] == y_f[use_f])
        corr[use_o] = (po[use_o] == y_o[use_o])

        correct += corr.sum().item()
        tot += x.size(0)

    return correct / max(1, tot)

@torch.no_grad()
def eval_per_level(model, loader):
    model.eval()
    tot_o = tot_f = tot_g = tot_s = 0
    cor_o = cor_f = cor_g = cor_s = 0

    for x, y_o, y_f, y_g, y_s, mask in loader:
        x = x.to(DEVICE)
        y_o = y_o.to(DEVICE)
        y_f = y_f.to(DEVICE)
        y_g = y_g.to(DEVICE)
        y_s = y_s.to(DEVICE)
        mask = mask.to(DEVICE)

        lo, lf, lg, ls = model(x)
        po = lo.argmax(1)
        pf = lf.argmax(1)
        pg = lg.argmax(1)
        ps = ls.argmax(1)

        mo = mask[:, 0] > 0.5
        mf = mask[:, 1] > 0.5
        mg = mask[:, 2] > 0.5
        ms = mask[:, 3] > 0.5

        if mo.sum() > 0:
            cor_o += (po[mo] == y_o[mo]).sum().item()
            tot_o += mo.sum().item()
        if mf.sum() > 0:
            cor_f += (pf[mf] == y_f[mf]).sum().item()
            tot_f += mf.sum().item()
        if mg.sum() > 0:
            cor_g += (pg[mg] == y_g[mg]).sum().item()
            tot_g += mg.sum().item()
        if ms.sum() > 0:
            cor_s += (ps[ms] == y_s[ms]).sum().item()
            tot_s += ms.sum().item()

    return {
        "order": cor_o / max(1, tot_o),
        "family": cor_f / max(1, tot_f),
        "genus": cor_g / max(1, tot_g),
        "species": cor_s / max(1, tot_s),
    }

# 8) Training loop
best_score = -1.0

for ep in range(1, EPOCHS + 1):
    student.train()
    running = 0.0

    for x, y_o, y_f, y_g, y_s, mask in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y_o = y_o.to(DEVICE, non_blocking=True)
        y_f = y_f.to(DEVICE, non_blocking=True)
        y_g = y_g.to(DEVICE, non_blocking=True)
        y_s = y_s.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        mo = mask[:, 0] > 0.5
        mf = mask[:, 1] > 0.5
        mg = mask[:, 2] > 0.5
        ms = mask[:, 3] > 0.5

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            with torch.no_grad():
                t_o, t_f, t_g, t_s = teacher(x)

            s_o, s_f, s_g, s_s = student(x)

            # Supervised loss
            # - order: global CE
            # - family/genus/species: branch-masked CE
            loss_sup_o = W_O * safe_ce(s_o, y_o, mo)

            loss_sup_f = W_F * branch_ce_loss(
                batch_logits=s_f[mf],
                batch_targets=y_f[mf],
                parent_targets=y_o[mf],
                child_map=order_to_family_idx
            ) if mf.sum() > 0 else torch.tensor(0.0, device=DEVICE)

            loss_sup_g = W_G * branch_ce_loss(
                batch_logits=s_g[mg],
                batch_targets=y_g[mg],
                parent_targets=y_f[mg],
                child_map=family_to_genus_idx
            ) if mg.sum() > 0 else torch.tensor(0.0, device=DEVICE)

            loss_sup_s = W_S * branch_ce_loss(
                batch_logits=s_s[ms],
                batch_targets=y_s[ms],
                parent_targets=y_g[ms],
                child_map=genus_to_species_idx
            ) if ms.sum() > 0 else torch.tensor(0.0, device=DEVICE)

            loss_sup = loss_sup_o + loss_sup_f + loss_sup_g + loss_sup_s

           
            # KD loss
            # - order: global KD
            # - family/genus/species: branch-masked KD
            loss_kd_o = W_O * safe_kd(s_o, t_o, mo, T=TEMP)

            loss_kd_f = W_F * branch_kd_loss(
                batch_s_logits=s_f[mf],
                batch_t_logits=t_f[mf],
                parent_targets=y_o[mf],
                child_map=order_to_family_idx,
                T=TEMP
            ) if mf.sum() > 0 else torch.tensor(0.0, device=DEVICE)

            loss_kd_g = W_G * branch_kd_loss(
                batch_s_logits=s_g[mg],
                batch_t_logits=t_g[mg],
                parent_targets=y_f[mg],
                child_map=family_to_genus_idx,
                T=TEMP
            ) if mg.sum() > 0 else torch.tensor(0.0, device=DEVICE)

            loss_kd_s = W_S * branch_kd_loss(
                batch_s_logits=s_s[ms],
                batch_t_logits=t_s[ms],
                parent_targets=y_g[ms],
                child_map=genus_to_species_idx,
                T=TEMP
            ) if ms.sum() > 0 else torch.tensor(0.0, device=DEVICE)

            loss_kd = loss_kd_o + loss_kd_f + loss_kd_g + loss_kd_s

            loss = loss_sup + LAMBDA_KD * loss_kd

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running += loss.item() * x.size(0)

    train_loss = running / max(1, len(train_loader.dataset))
    val_best = eval_best_available(student, val_loader)
    val_levels = eval_per_level(student, val_loader)

    print(
        f"Epoch {ep:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_best={val_best:.4f} | "
        f"order={val_levels['order']:.4f} | "
        f"family={val_levels['family']:.4f} | "
        f"genus={val_levels['genus']:.4f} | "
        f"species={val_levels['species']:.4f}"
    )

    if val_best > best_score:
        best_score = val_best
        torch.save({
            "model": student.state_dict(),
            "order2idx": order2idx_n,
            "family2idx": family2idx_n,
            "genus2idx": genus2idx_n,
            "species2idx": species2idx_n,
            "species_to_genus": species_to_genus,
            "genus_to_family": genus_to_family,
            "family_to_order": family_to_order,
            "order_to_family_idx": dict(order_to_family_idx),
            "family_to_genus_idx": dict(family_to_genus_idx),
            "genus_to_species_idx": dict(genus_to_species_idx),
            "note": (
                f"Student branch-masked KD on DongVan + replay. "
                f"KD={LAMBDA_KD}, T={TEMP}, "
                f"supervised=branch-masked(family/genus/species), "
                f"trainable=layer4+heads."
            )
        }, SAVE_PATH)
        print("  -> saved best:", SAVE_PATH)

print("\nDONE")
print("Best val(best-available-acc):", best_score)
print("Saved:", SAVE_PATH)

# Load best model
best_ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
student.load_state_dict(best_ckpt["model"])

# Final evaluation on test set
test_best = eval_best_available(student, test_loader)
test_levels = eval_per_level(student, test_loader)

print("\nTEST")
print("test_best =", test_best)
print("order     =", test_levels["order"])
print("family    =", test_levels["family"])
print("genus     =", test_levels["genus"])
print("species   =", test_levels["species"])

Tax rows | vnua spiece: 53 dv spiece: 22 merged: 69
DongVan dataset total images: 2237
Train: 1565
Val: 335
Test: 337
Train=2347 (DV=1565 + replay=782)
Val = 335
Test = 337
[Dataset stats] {'species': 931, 'genus': 73, 'family': 846, 'order': 497, 'skip': 0}
[Dataset stats] {'species': 23, 'genus': 8, 'family': 177, 'order': 127, 'skip': 0}
[Dataset stats] {'species': 31, 'genus': 7, 'family': 191, 'order': 108, 'skip': 0}
Prepared loaders: train= 2347 val= 335 test= 337 ckpt= /kaggle/input/models/ichbingiangthanh/model-vnua-080626/pytorch/default/1/trained_model_vnua_base_080626.pt
Device: cuda
Branch map stats:
  orders with family children : 25
  families with genus children: 39
  genera with species children: 51
Loaded teacher/student from: /kaggle/input/models/ichbingiangthanh/model-vnua-080626/pytorch/default/1/trained_model_vnua_base_080626.pt
Trainable params: 8,479,399 / 11,262,183


/tmp/ipykernel_58/735164139.py:549: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
/tmp/ipykernel_58/735164139.py:655: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Epoch 01/8 | train_loss=2.1989 | val_best=0.2896 | order=0.4537 | family=0.2692 | genus=0.4516 | species=0.2174
  -> saved best: /kaggle/working/model_distillation_080626.pt
Epoch 02/8 | train_loss=1.6086 | val_best=0.3015 | order=0.5463 | family=0.2212 | genus=0.2903 | species=0.1304
  -> saved best: /kaggle/working/model_distillation_080626.pt
Epoch 03/8 | train_loss=1.4667 | val_best=0.3134 | order=0.5672 | family=0.2692 | genus=0.3226 | species=0.0870
  -> saved best: /kaggle/working/model_distillation_080626.pt
Epoch 04/8 | train_loss=1.4449 | val_best=0.3373 | order=0.6000 | family=0.3221 | genus=0.4839 | species=0.0870
  -> saved best: /kaggle/working/model_distillation_080626.pt
Epoch 05/8 | train_loss=1.4479 | val_best=0.3761 | order=0.6448 | family=0.3173 | genus=0.5806 | species=0.0870
  -> saved best: /kaggle/working/model_distillation_080626.pt
Epoch 06/8 | train_loss=1.3623 | val_best=0.3672 | order=0.6567 | family=0.2596 | genus=0.3548 | species=0.0000
Epoch 07/8 | train